<a href="https://colab.research.google.com/github/AliefReefandi/Grokking-Deep-Learning/blob/main/Chapter_08_Learning_Signal_and_Ignoring_Noise.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Chapter 8 — Learning Signal and Ignoring Noise

## Tujuan Pembelajaran

Pada chapter ini kita mempelajari bagaimana neural network dapat
belajar pola yang penting dari data dan mengurangi pengaruh noise.

Setelah menyelesaikan chapter ini, kita diharapkan dapat:

- Memahami perbedaan training dan testing.
- Memahami memorization dan generalization.
- Memahami overfitting.
- Memahami konsep regularization.
- Memahami early stopping.
- Memahami dropout.
- Memahami bagaimana dropout membantu mengurangi overfitting.
- Memahami batch gradient descent.
- Mengimplementasikan dropout dan batch training menggunakan NumPy.

## 2. Three-Layer Network on MNIST

Pada chapter ini neural network kembali digunakan untuk melakukan
klasifikasi dataset MNIST.

MNIST berisi gambar angka tulisan tangan.

Setiap gambar berukuran:

28 × 28 pixel

Sehingga setiap gambar dapat diubah menjadi:

784 input

Network yang digunakan memiliki:

Input Layer → Hidden Layer → Output Layer

Output layer memiliki 10 nilai yang mewakili angka:

0, 1, 2, 3, 4, 5, 6, 7, 8, 9

In [1]:
import numpy as np
from keras.datasets import mnist

(x_train, y_train), (x_test, y_test) = mnist.load_data()

images = x_train[:1000].reshape(1000, 28 * 28) / 255
labels = y_train[:1000]

test_images = x_test.reshape(len(x_test), 28 * 28) / 255
test_labels = y_test

print("Training images:", images.shape)
print("Testing images :", test_images.shape)

11490434/11490434 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Training images: (1000, 784)
Testing images : (10000, 784)


## 3. Training vs Testing

Training data adalah data yang digunakan network untuk belajar.

Testing data adalah data yang tidak digunakan saat training dan
digunakan untuk menguji kemampuan network terhadap data baru.

Contohnya:

Training:
Network melihat 1.000 gambar MNIST.

Testing:
Network diberikan gambar yang belum pernah digunakan saat training.

Training accuracy yang tinggi belum tentu berarti network bagus.

Yang lebih penting adalah kemampuan network melakukan prediksi
pada data yang belum pernah dilihat.

## 4. Memorization vs Generalization

### Memorization

Network menghafal pola spesifik dari training data.

### Generalization

Network mempelajari pola yang lebih umum sehingga dapat bekerja
pada data baru.

Tujuan deep learning bukan hanya mendapatkan training accuracy tinggi.

Tujuannya adalah:

Training Data → Belajar pola
                    ->
               Generalization
                    ->
              Data baru → Prediction

## 5. Overfitting

Overfitting terjadi ketika neural network terlalu cocok dengan
training data sehingga performanya pada data baru menurun.

Contoh:

Training Accuracy → terus naik
Testing Accuracy  → mulai turun

Artinya network semakin baik dalam menghafal training data,
tetapi semakin buruk dalam melakukan generalisasi.

Overfitting lebih mudah terjadi pada neural network yang memiliki
expressive power besar, misalnya lebih banyak layer dan weight.

## 6. Regularization

Regularization adalah kumpulan metode yang digunakan untuk membantu
model melakukan generalization dan mengurangi overfitting.

Tujuannya adalah:

Belajar signal
+
Mengabaikan noise

Signal adalah pola penting yang membantu prediksi.

Noise adalah detail yang tidak relevan tetapi dapat dipelajari
oleh network jika terlalu lama atau terlalu bebas melakukan training.

## 7. Early Stopping

Early stopping adalah bentuk regularization yang sangat sederhana.

Ide utamanya:

"Stop training ketika performa pada validation data mulai memburuk."

Pada awal training:

Training Error ↓

Validation Error ↓

Tetapi setelah terlalu lama:

Training Error ↓

Validation Error ↑

Ketika validation performance mulai memburuk,
training dapat dihentikan.

## 8. Dropout

Dropout adalah teknik regularization dengan cara mematikan neuron
secara acak selama training.

Misalnya:

Hidden Layer:

[1] [1] [1] [1] [1] [1]

Setelah dropout:

[1] [0] [1] [0] [1] [0]

Neuron yang bernilai 0 tidak digunakan pada training tersebut.

Dengan demikian network tidak dapat terlalu bergantung pada
neuron tertentu.

## 9. Mengapa Dropout Bekerja?

Dropout membuat network besar seolah-olah menjadi network yang lebih
kecil pada setiap proses training.

Network besar memiliki kemampuan yang besar untuk mempelajari
detail dan noise.

Dengan dropout, hanya sebagian neuron yang aktif pada setiap
training step.

Hal ini membuat network lebih sulit menghafal noise.

Buku juga menghubungkan dropout dengan konsep ensemble:
banyak subnetworks yang berbeda dapat belajar dari network yang sama.

## 10. Implementasi Dropout

Pada implementasi buku, dropout menggunakan sebuah dropout mask
berisi angka 0 dan 1.

Mask tersebut digunakan untuk mematikan neuron pada hidden layer.

Jika neuron mendapatkan:

0 → neuron dimatikan
1 → neuron tetap aktif

Buku menggunakan dropout 50%, sehingga rata-rata setengah neuron
dimatikan selama training.

In [2]:
import numpy as np

layer_1 = np.array([[1.0, 2.0, 3.0, 4.0]])

dropout_mask = np.random.randint(
    2,
    size=layer_1.shape
)

layer_1 *= dropout_mask * 2

print("Dropout mask:")
print(dropout_mask)

print("\nLayer 1 setelah dropout:")
print(layer_1)

Dropout mask:
[[1 1 1 1]]

Layer 1 setelah dropout:
[[2. 4. 6. 8.]]


### Mengapa dikali 2?

Karena dropout yang digunakan adalah 50%.

Jika setengah neuron dimatikan, nilai rata-rata layer akan turun
sekitar setengah.

Untuk menjaga skala output tetap sama antara training dan testing,
nilai neuron yang tetap aktif dikalikan:

1 / 0.5 = 2

## 11. Evaluasi Dropout pada MNIST

Tanpa dropout, network dapat mencapai training accuracy yang sangat
tinggi tetapi kemudian mengalami penurunan test accuracy karena
overfitting.

Dengan dropout:

- Training menjadi lebih sulit.
- Training accuracy meningkat lebih lambat.
- Network lebih tahan terhadap overfitting.
- Test accuracy menjadi lebih stabil.

## 12. Batch Gradient Descent

Sebelumnya training dilakukan satu data pada satu waktu.

Pada batch gradient descent, beberapa data diproses sekaligus.

Contoh pada buku:

100 training examples → satu batch

Weight update kemudian dihitung berdasarkan rata-rata dari
weight update seluruh contoh dalam batch tersebut.

Keuntungannya:

- Training lebih cepat.
- Update weight menjadi lebih stabil.
- Proses learning menjadi lebih halus.

In [3]:
batch_size = 100

for i in range(int(len(images) / batch_size)):

    batch_start = i * batch_size
    batch_end = (i + 1) * batch_size

    layer_0 = images[batch_start:batch_end]

    print(
        "Batch:",
        i + 1,
        "| Data:",
        batch_start,
        "-",
        batch_end
    )

Batch: 1 | Data: 0 - 100
Batch: 2 | Data: 100 - 200
Batch: 3 | Data: 200 - 300
Batch: 4 | Data: 300 - 400
Batch: 5 | Data: 400 - 500
Batch: 6 | Data: 500 - 600
Batch: 7 | Data: 600 - 700
Batch: 8 | Data: 700 - 800
Batch: 9 | Data: 800 - 900
Batch: 10 | Data: 900 - 1000


Pada implementasi neural network, batch kemudian digunakan untuk
menghitung layer, error, delta, dan weight update secara bersamaan.

Dengan batch berisi 100 data, operasi matrix dapat memproses banyak
dot product sekaligus.

Hal ini membuat training lebih cepat dibandingkan memproses satu
data pada satu waktu.

In [6]:
import numpy as np
from keras.datasets import mnist

# =========================================================
# 1. LOAD DATASET MNIST
# =========================================================

np.random.seed(1)

(x_train, y_train), (x_test, y_test) = mnist.load_data()


# =========================================================
# 2. PREPROCESSING DATA
# =========================================================

# Mengambil 1000 data training
images = x_train[:1000].reshape(1000, 28 * 28) / 255
labels = y_train[:1000]

# One-hot encoding untuk label training
one_hot_labels = np.zeros((len(labels), 10))

for i, l in enumerate(labels):
    one_hot_labels[i][l] = 1

labels = one_hot_labels


# Data testing
test_images = x_test.reshape(len(x_test), 28 * 28) / 255

# One-hot encoding untuk label testing
test_labels = np.zeros((len(y_test), 10))

for i, l in enumerate(y_test):
    test_labels[i][l] = 1


print("Training images :", images.shape)
print("Training labels :", labels.shape)
print("Testing images  :", test_images.shape)
print("Testing labels  :", test_labels.shape)


# =========================================================
# 3. FUNGSI RELU
# =========================================================

def relu(x):
    return (x >= 0) * x


def relu2deriv(output):
    return output >= 0


# =========================================================
# 4. HYPERPARAMETER
# =========================================================

batch_size = 100

alpha = 0.001
iterations = 300

pixels_per_image = 784
num_labels = 10
hidden_size = 100


# =========================================================
# 5. INISIALISASI WEIGHTS
# =========================================================

weights_0_1 = (
    0.2 * np.random.random(
        (pixels_per_image, hidden_size)
    ) - 0.1
)

weights_1_2 = (
    0.2 * np.random.random(
        (hidden_size, num_labels)
    ) - 0.1
)


# =========================================================
# 6. TRAINING BATCH + DROPOUT
# =========================================================

for j in range(iterations):

    error = 0.0
    correct_cnt = 0

    # Membagi data menjadi beberapa batch
    for i in range(int(len(images) / batch_size)):

        batch_start = i * batch_size
        batch_end = (i + 1) * batch_size

        # -------------------------------------------------
        # Layer 0: Input
        # -------------------------------------------------

        layer_0 = images[batch_start:batch_end]


        # -------------------------------------------------
        # Layer 1: Hidden Layer
        # -------------------------------------------------

        layer_1 = relu(
            np.dot(
                layer_0,
                weights_0_1
            )
        )


        # -------------------------------------------------
        # DROPOUT
        # -------------------------------------------------

        dropout_mask = np.random.randint(
            2,
            size=layer_1.shape
        )

        layer_1 *= dropout_mask * 2


        # -------------------------------------------------
        # Layer 2: Output
        # -------------------------------------------------

        layer_2 = np.dot(
            layer_1,
            weights_1_2
        )


        # -------------------------------------------------
        # Menghitung Error
        # -------------------------------------------------

        error += np.sum(
            (
                labels[batch_start:batch_end]
                - layer_2
            ) ** 2
        )


        # -------------------------------------------------
        # Menghitung Accuracy
        # -------------------------------------------------

        for k in range(batch_size):

            correct_cnt += int(
                np.argmax(
                    layer_2[k:k+1]
                )
                ==
                np.argmax(
                    labels[
                        batch_start + k:
                        batch_start + k + 1
                    ]
                )
            )


        # -------------------------------------------------
        # Backpropagation
        # -------------------------------------------------

        layer_2_delta = (
            labels[batch_start:batch_end]
            - layer_2
        ) / batch_size


        layer_1_delta = (
            layer_2_delta.dot(
                weights_1_2.T
            )
            * relu2deriv(layer_1)
        )


        # Mengembalikan efek dropout
        layer_1_delta *= dropout_mask


        # -------------------------------------------------
        # Update Weights
        # -------------------------------------------------

        weights_1_2 += (
            alpha
            * layer_1.T.dot(
                layer_2_delta
            )
        )

        weights_0_1 += (
            alpha
            * layer_0.T.dot(
                layer_1_delta
            )
        )


    # =====================================================
    # 7. MENAMPILKAN HASIL TRAINING
    # =====================================================

    if j % 10 == 0:

        print(
            "Iteration:",
            j,
            "| Error:",
            round(error, 4),
            "| Accuracy:",
            round(
                correct_cnt / float(len(images)),
                4
            )
        )

Training images : (1000, 784)
Training labels : (1000, 10)
Testing images  : (10000, 784)
Testing labels  : (10000, 10)
Iteration: 0 | Error: 2080.2839 | Accuracy: 0.084
Iteration: 10 | Error: 1534.3039 | Accuracy: 0.097
Iteration: 20 | Error: 1347.7208 | Accuracy: 0.147
Iteration: 30 | Error: 1268.6832 | Accuracy: 0.147
Iteration: 40 | Error: 1225.7491 | Accuracy: 0.141
Iteration: 50 | Error: 1127.6877 | Accuracy: 0.19
Iteration: 60 | Error: 1102.5784 | Accuracy: 0.213
Iteration: 70 | Error: 1062.8363 | Accuracy: 0.221
Iteration: 80 | Error: 1028.1557 | Accuracy: 0.253
Iteration: 90 | Error: 992.7664 | Accuracy: 0.271
Iteration: 100 | Error: 975.8329 | Accuracy: 0.28
Iteration: 110 | Error: 937.6997 | Accuracy: 0.312
Iteration: 120 | Error: 923.2136 | Accuracy: 0.316
Iteration: 130 | Error: 910.1055 | Accuracy: 0.33
Iteration: 140 | Error: 883.6785 | Accuracy: 0.347
Iteration: 150 | Error: 866.3946 | Accuracy: 0.376
Iteration: 160 | Error: 865.5532 | Accuracy: 0.376
Iteration: 170 | E

## 15. Rangkuman

Chapter 8 membahas bagaimana neural network dapat belajar signal
tanpa terlalu banyak mempelajari noise.

Konsep utama:

### Overfitting
Model terlalu cocok dengan training data dan buruk pada data baru.

### Generalization
Kemampuan model bekerja dengan baik pada data yang belum pernah dilihat.

### Regularization
Teknik untuk membantu model melakukan generalization.

### Early Stopping
Menghentikan training ketika performa validation mulai memburuk.

### Dropout
Mematikan neuron secara acak selama training.

### Batch Gradient Descent
Memproses beberapa data sekaligus dan menggunakan rata-rata
weight update.

Alur sederhananya:

Training
   ↓

Belajar Signal
   ↓

Kurangi Noise
   ↓

Generalization
   ↓
   
Prediksi lebih baik pada Data Baru

## 16. Kesimpulan

Chapter 8 menjelaskan bahwa neural network yang sangat baik pada
training data belum tentu memiliki performa yang baik pada data baru.

Masalah tersebut disebut overfitting.

Untuk mengatasinya, digunakan regularization seperti early stopping
dan dropout.

Dropout bekerja dengan mematikan neuron secara acak sehingga network
tidak terlalu bergantung pada bagian tertentu dari arsitektur.

Chapter ini juga memperkenalkan batch gradient descent yang
memproses beberapa data sekaligus sehingga proses training menjadi
lebih cepat dan weight update menjadi lebih stabil.

Dengan demikian, tujuan training bukan hanya mendapatkan error
training yang kecil, tetapi menghasilkan model yang mampu
melakukan generalization terhadap data baru.